In [1]:
"""
Supplementary Note 2: Center Weighting

Authors: Eugene Oh, Annemarie Becker

inputFile:
.map file generated by Bowtie default output.

outputFileP:
read density file for plus strand
    col0: position along genome
    col1: read density at that position

outputFileM:
read density file for minus strand
    col0: position along genome
    col1: read density at that position

"""
def rawdata(inputFile, outputFileP, outputFileM):
    
    pDict = {}
    mDict = {}
    
    inFile = open(inputFile, 'r')
    line = inFile.readline()
    while line != '':
        fields = line.split()
        col2 = str(fields[2])   #strand; note: if sequencing was performed without barcode reading, the column numbering is changed
        col4 = int(fields[4])   #left-most position
        col5 = str(fields[5])   #footprint seq
        length = len(col5)      #footprint length
                
        if 22 < length < 42:    #select range of footprint read lengths
            if col2 == '+':	#for plus strand
                columns = len(fields)   #count number of columns to check if alignment contains mismatches
                if columns > 8:		
                    col8 = str(fields[8])
                    if col8.startswith("0"):	#if there is a mismatch in the 1st position
                        length0 = length - 1    #subtract wrong base at 1st position
                        end5 = col4 + 2		#Bowtie uses 0-based offset: transform to 1-based and subtract 1st base
                        end3 = end5 + length0 - 1
                        centerEnd5 = end5 + 11	#define center
                        centerEnd3 = end3 - 11
                        centerLength = centerEnd3 - centerEnd5 + 1
                    else:
                        end5 = col4 + 1     #Bowtie uses zero-based offset, transform to 1-based
                        end3 = end5 + length - 1
                        centerEnd5 = end5 + 11
                        centerEnd3 = end3 - 11
                        centerLength = centerEnd3 - centerEnd5 + 1
                else:
                    end5 = col4 + 1
                    end3 = end5 + length - 1
                    centerEnd5 = end5 + 11
                    centerEnd3 = end3 - 11
                    centerLength = centerEnd3 - centerEnd5 + 1

                for elem in range(centerEnd5, centerEnd3 + 1):
                    if elem in pDict:
                        pDict[elem] += (1.0 / centerLength)
                    else:
                        pDict[elem] = (1.0 / centerLength) 

            elif col2 == '-':		#for minus strand
                columns = len(fields)
                if columns > 8:
                    col8 = str(fields[8])
                    if col8.startswith("0"):
                        length0 = length - 1
                        end3 = col4 + 1         #for minus strand, Bowtie gives leftmost position (3' end) with zero-based numbering
                        end5 = end3 + length0 - 1
                        centerEnd5 = end5 - 11
                        centerEnd3 = end3 + 11          
                        centerLength = centerEnd5 - centerEnd3 + 1
                    else:
                        end3 = col4 + 1
                        end5 = end3 + length - 1
                        centerEnd5 = end5 - 11
                        centerEnd3 = end3 + 11            
                        centerLength = centerEnd5 - centerEnd3 + 1
                else: 
                    end3 = col4 + 1
                    end5 = end3 + length - 1
                    centerEnd5 = end5 - 11
                    centerEnd3 = end3 + 11            
                    centerLength = centerEnd5 - centerEnd3 + 1
                
                for elem in range(centerEnd3, centerEnd5 + 1):
                    if elem in mDict:
                        mDict[elem] += (1.0 / centerLength)
                    else:
                        mDict[elem] = (1.0 / centerLength)

        line = inFile.readline()
    
    pList = list(pDict.items())
    pList.sort()
    outFileP = open(outputFileP, 'w')
    for J in pList:
        outFileP.write(str(J[0]) + '\t' + str(J[1]) + '\n')        

    mList = list(mDict.items())
    mList.sort()
    outFileM = open(outputFileM, 'w')
    for J in mList:
        outFileM.write(str(J[0]) + '\t' + str(J[1]) + '\n')
    
    inFile.close()
    outFileP.close()
    outFileM.close()

In [7]:
# sample name
sample = ['Ssb1-inter-rep1.map','Ssb1-inter-rep2.map','Ssb2-inter-rep1.map','Ssb2-inter-rep2.map',
          'Ssb1-trans-rep1.map','Ssb1-trans-rep2.map','Ssb2-trans-rep1.map','Ssb2-trans-rep2.map']

outPlus = ['Ssb1-inter-rep1.PDensity.txt','Ssb1-inter-rep2.PDensity.txt','Ssb2-inter-rep1.PDensity.txt','Ssb2-inter-rep2.PDensity.txt',
          'Ssb1-trans-rep1.PDensity.txt','Ssb1-trans-rep2.PDensity.txt','Ssb2-trans-rep1.PDensity.txt','Ssb2-trans-rep2.PDensity.txt']

outMinus = ['Ssb1-inter-rep1.MDensity.txt','Ssb1-inter-rep2.MDensity.txt','Ssb2-inter-rep1.MDensity.txt','Ssb2-inter-rep2.MDensity.txt',
          'Ssb1-trans-rep1.MDensity.txt','Ssb1-trans-rep2.MDensity.txt','Ssb2-trans-rep1.MDensity.txt','Ssb2-trans-rep2.MDensity.txt']

# run
for i in range(0,8):
    rawdata(sample[i],''.join(['2.ribo-density-data/',outPlus[i]]),''.join(['2.ribo-density-data/',outMinus[i]]))